## Inferece notebook for trained models

In [1]:
import os
from glob import glob
import matplotlib.pyplot as plt
import numpy as np
import PIL
import cv2
import tensorflow as tf
from tensorflow.keras.layers import Dense, Flatten
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras import layers, Model

from typing import List, Tuple

In [2]:
model_path = 'weights/resnet_101_320x320.h5'

In [8]:
data_dir = 'Malware_as_images/data_for_training/'

image_batches = tf.keras.preprocessing.image_dataset_from_directory(data_dir,
                                                      seed = 123,
                                                      image_size = (320, 320),
                                                      batch_size = 8)

image_batches = iter(image_batches)

Found 1042 files belonging to 2 classes.


In [9]:
images, labels = next(image_batches)

In [10]:
images.shape

TensorShape([8, 320, 320, 3])

In [11]:
model = load_model(model_path)
extractor = Model(inputs = model.inputs,
                        outputs = [model.layers[-2].output])
for images, labels in image_batches:
#     print(images)
#     print(labels)
    features = extractor(images)
    print(features.shape)
    break

(8, 256)


In [20]:
model = load_model(model_path)
extractor = Model(inputs = model.inputs,
                        outputs = [model.layers[-2].output])

img_path = test_images[1]
print(img_path)

image = cv2.imread(img_path)
image_resized = cv2.resize(image, (224, 224))
image = np.expand_dims(image_resized, axis = 0)
features = extractor(image)

# print(features)
print(features.shape)

Malware_as_images/processed_data/benign/benign_1.png
tf.Tensor(
[[15.399913    0.          0.          0.          0.          0.
   0.          0.          0.          0.          0.          0.
   0.          0.          0.          0.          4.4832296   0.
   0.          0.          0.          0.          0.          0.
   0.          0.          0.          0.          1.2783636   0.
   0.          0.          0.          0.          0.          0.
   0.          5.9685664   0.          0.          0.          0.
   0.          0.          0.          0.          0.          0.
   0.          0.          0.         13.021573    0.          0.
   0.          0.          0.          0.          0.          0.
   0.          0.          0.          0.          0.          0.
   5.477667    0.          0.          0.          0.          0.
   0.          0.          0.          0.          0.          0.
   0.          0.          3.746981    0.          0.          9.583257
   0. 

In [2]:
class MalwareDetection:
    """
        This class is an inference for trained models on malware images data
    """
    def __init__(self, model_path : str, optimizer, loss_fn : str, 
                         metrics : List[str], input_shape : Tuple):
        self.model_path = model_path
        self.optimizer = optimizer
        self.loss = loss_fn
        self.metrics = metrics
        self.input_shape = input_shape
        self.model = load_model(self.model_path)
        self.classes = ["benign", "malicious"]
        
    def load_image(self, img_path):
        image = cv2.imread(img_path)
        image_resized = cv2.resize(image, self.input_shape)
        image = np.expand_dims(image_resized, axis = 0)
        print("image load successfully")
        print(img_path)
        return image
    
    def check_malware_image(self, img_path):
        img = self.load_image(img_path)
        out = self.model.predict(img)[0]
        pred = self.classes[np.argmax(list(out))]
        return f"predicted class is : {pred}"
    
    def get_embeddings(self, img_path):
        """
            This method is used to get second last layers embeddings
        """
        img = self.load_image(img_path)
        extractor = Model(inputs = model.inputs,
                          outputs = [model.layers[-2].output])
        emb = extractor.predict(img)
        return emb

In [3]:
test_images = glob("Malware_as_images/processed_data/**/*.png")
print(len(test_images))

1099


In [4]:
clf = MalwareDetection(model_path = 'weights/resnet_50_224x224.h5',
                          optimizer = Adam(lr = 0.001),
                          loss_fn = 'sparse_categorical_crossentropy',
                          metrics = ['accuracy'],
                          input_shape = (224, 224))


In [9]:
n = np.random.randint(0, len(test_images))

img_path = test_images[n]    # selecting a random image
clf.check_malware_image(img_path)

image load successfully
Malware_as_images/processed_data/malicious/malicious_498.png


'predicted class is : malicious'